# 14 — Final Fine-tuning (spoof head freeze)

> **배경:**  
> 13번 Fine-tuning 결과 binary head는 웹캠 도메인 적응 성공(98%)했으나  
> spoof head가 웹캠 이미지를 전부 Live(0)으로 과적합됨  
> → FAKE인데 spoof_type=Live 모순 발생
>
> **해결 전략:**  
> `spoof` 레이어 완전 freeze → `binary` + `shared` 레이어만 재학습  
> spoof head는 `stage2_best.h5` 원본 가중치 그대로 유지
>
> **산출물:** `models/stage2_final.h5`
>
> **체크리스트:**
> - [ ] Cell 0: Drive 마운트
> - [ ] Cell 1: 경로 설정
> - [ ] Cell 2: 레이어 freeze 설정 확인
> - [ ] Cell 3: 데이터 준비
> - [ ] Cell 4: Fine-tuning 실행
> - [ ] Cell 5: 검증 (binary + spoof 둘 다 확인)
> - [ ] Cell 6: xai_explainer.py 경로 업데이트

## Cell 0 — Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive 마운트 완료')

## Cell 1 — 경로 설정

In [ ]:
import os, json
import numpy as np
import cv2
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

BASE        = '/content/drive/MyDrive/face-anti-spoofing'
MODEL_PATH  = f'{BASE}/models/stage2_best.h5'   # 원본 모델
FINAL_MODEL = f'{BASE}/models/stage2_final.h5'  # 최종 저장
WEBCAM_DIR  = f'{BASE}/data/webcam_live'
CROP_DIR    = f'{BASE}/data/cropped'
REPORT_DIR  = f'{BASE}/reports/phase5'

os.makedirs(REPORT_DIR, exist_ok=True)

print('TF :', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('웹캠 이미지:', len(list(Path(WEBCAM_DIR).glob('*.jpg'))), '장')

## Cell 2 — 레이어 Freeze 설정

> **핵심:** `spoof` 레이어만 완전 freeze  
> `binary`, `shared`, `dense`, `dense_1` 등 binary 경로 레이어만 학습

In [ ]:
# 원본 모델 로드
model = tf.keras.models.load_model(MODEL_PATH)
print('✅ 원본 모델 로드 완료')
print('  레이어 수:', len(model.layers))

# spoof head 완전 freeze
# binary 경로 레이어만 trainable
FREEZE_LAYERS  = ['spoof', 'dense_1']           # spoof head 고정
TRAIN_LAYERS   = ['binary', 'shared', 'dense',  # binary 경로 학습
                  'dropout_1', 'batch_normalization',
                  'global_average_pooling2d']

# 기본: 전체 freeze
for layer in model.layers:
    layer.trainable = False

# binary 경로만 unfreeze
for layer in model.layers:
    if any(t in layer.name for t in TRAIN_LAYERS):
        layer.trainable = True

# 결과 확인
print('\n=== 레이어별 trainable 상태 (마지막 15개) ===')
for layer in model.layers[-15:]:
    status = '🟢 학습' if layer.trainable else '🔴 고정'
    print(f'  {status}  {layer.name}')

trainable   = sum(1 for l in model.layers if l.trainable)
frozen      = sum(1 for l in model.layers if not l.trainable)
print(f'\n학습 레이어: {trainable} / 고정 레이어: {frozen}')

# spoof 레이어 freeze 확인
spoof_layer = model.get_layer('spoof')
print(f'spoof layer trainable: {spoof_layer.trainable}  ← False여야 정상')

In [ ]:
# 재컴파일 — binary loss만 사용
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        'binary': 'binary_crossentropy',
        'spoof':  'sparse_categorical_crossentropy',
    },
    loss_weights={'binary': 1.0, 'spoof': 0.0},  # spoof loss 완전 무시
    metrics={'binary': 'accuracy', 'spoof': 'accuracy'}
)
print('✅ 재컴파일 완료 (lr=1e-5, spoof loss_weight=0.0)')

## Cell 3 — 데이터 준비

In [ ]:
def load_and_preprocess(img_path, size=224):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    return img.astype('float32') / 255.0

X, y_binary, y_spoof = [], [], []

# 웹캠 Live
webcam_imgs = sorted(Path(WEBCAM_DIR).glob('*.jpg'))
print(f'웹캠 Live: {len(webcam_imgs)}장')
for p in webcam_imgs:
    img = load_and_preprocess(p)
    if img is not None:
        X.append(img)
        y_binary.append(0)
        y_spoof.append(0)

# 기존 Live (1:1)
existing_live = sorted(Path(f'{CROP_DIR}/live').glob('*.jpg'))[:len(webcam_imgs)]
print(f'기존 Live: {len(existing_live)}장')
for p in existing_live:
    img = load_and_preprocess(p)
    if img is not None:
        X.append(img)
        y_binary.append(0)
        y_spoof.append(0)

# Spoof (Live와 동수)
n_live = len(X)
spoof_per_cat = n_live // 3
for cat, type_idx in [('print', 1), ('replay', 2), ('mask', 3)]:
    spoof_imgs = sorted(Path(f'{CROP_DIR}/{cat}').glob('*.jpg'))[:spoof_per_cat]
    print(f'{cat}: {len(spoof_imgs)}장')
    for p in spoof_imgs:
        img = load_and_preprocess(p)
        if img is not None:
            X.append(img)
            y_binary.append(1)
            y_spoof.append(type_idx)

X        = np.array(X)
y_binary = np.array(y_binary, dtype='float32')
y_spoof  = np.array(y_spoof,  dtype='int32')

print(f'\n총 데이터: {len(X)}장')
print(f'  REAL: {(y_binary==0).sum()}  FAKE: {(y_binary==1).sum()}')

idx = np.arange(len(X))
tr_idx, val_idx = train_test_split(idx, test_size=0.2, stratify=y_binary, random_state=42)
X_tr,  X_val  = X[tr_idx],        X[val_idx]
yb_tr, yb_val = y_binary[tr_idx], y_binary[val_idx]
ys_tr, ys_val = y_spoof[tr_idx],  y_spoof[val_idx]

print(f'Train: {len(X_tr)}  Val: {len(X_val)}')

## Cell 4 — Fine-tuning 실행

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        FINAL_MODEL,
        monitor='val_binary_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_binary_accuracy',
        patience=5,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    )
]

history = model.fit(
    X_tr,
    {'binary': yb_tr, 'spoof': ys_tr},
    validation_data=(X_val, {'binary': yb_val, 'spoof': ys_val}),
    epochs=20,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Fine-tuning 완료')
print('   저장:', FINAL_MODEL)

In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['binary_accuracy'],     label='Train')
axes[0].plot(history.history['val_binary_accuracy'], label='Val')
axes[0].set_title('Binary Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Final Fine-tuning 학습 곡선 (spoof head freeze)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/final_finetune_curve.png', dpi=150)
plt.show()
print('✅ 저장:', f'{REPORT_DIR}/final_finetune_curve.png')

## Cell 5 — 검증

> **확인 포인트:**
> 1. 웹캠 Live → REAL 판정 (binary 정상)
> 2. Spoof 이미지 → 올바른 유형 분류 (spoof head 원본 유지)

In [ ]:
final_model = tf.keras.models.load_model(FINAL_MODEL)
print('✅ 최종 모델 로드:', FINAL_MODEL)

SPOOF_KO = {0: 'Live', 1: 'Print', 2: 'Replay', 3: 'Mask'}

def predict_both(model, img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224)).astype('float32') / 255.0
    inp = np.expand_dims(img, 0)
    preds = model.predict(inp, verbose=0)
    binary_prob = float(preds[0][0][0])
    spoof_type  = int(np.argmax(preds[1][0]))
    verdict     = 'FAKE' if binary_prob >= 0.5 else 'REAL'
    return verdict, binary_prob, spoof_type

print('\n=== 웹캠 Live (REAL이어야 함) ===')
webcam_imgs = sorted(Path(WEBCAM_DIR).glob('*.jpg'))[:10]
webcam_real = 0
for p in webcam_imgs:
    img = cv2.imread(str(p))
    if img is None: continue
    verdict, prob, stype = predict_both(final_model, img)
    ok = '✅' if verdict == 'REAL' else '❌'
    print(f'  {ok} {p.name}: {verdict} ({prob:.1%})  spoof_type={SPOOF_KO.get(stype, stype)}')
    if verdict == 'REAL': webcam_real += 1

print(f'\n  웹캠 REAL 판정률: {webcam_real}/{len(webcam_imgs)} ({webcam_real/len(webcam_imgs):.0%})')

print('\n=== Spoof 이미지 (유형 올바르게 분류되어야 함) ===')
for cat, expected_type, expected_name in [
    ('print',  1, 'Print'),
    ('replay', 2, 'Replay'),
    ('mask',   3, 'Mask')
]:
    imgs = sorted(Path(f'{CROP_DIR}/{cat}').glob('*.jpg'))[:5]
    correct = 0
    for p in imgs:
        img = cv2.imread(str(p))
        if img is None: continue
        verdict, prob, stype = predict_both(final_model, img)
        if verdict == 'FAKE' and stype == expected_type:
            correct += 1
    print(f'  {cat}: FAKE+올바른유형 {correct}/{len(imgs)}')

## Cell 6 — xai_explainer.py 경로 업데이트

In [ ]:
XAI_PATH = f'{BASE}/src/xai_explainer.py'

with open(XAI_PATH) as f:
    code = f.read()

# stage2_webcam.h5 → stage2_final.h5
code = code.replace(
    'MODEL_PATH   = os.path.join(BASE, "models", "stage2_webcam.h5")',
    'MODEL_PATH   = os.path.join(BASE, "models", "stage2_final.h5")'
)
# stage2_best.h5 → stage2_final.h5 (혹시 원본으로 돌아간 경우)
code = code.replace(
    'MODEL_PATH   = os.path.join(BASE, "models", "stage2_best.h5")',
    'MODEL_PATH   = os.path.join(BASE, "models", "stage2_final.h5")'
)

with open(XAI_PATH, 'w') as f:
    f.write(code)

# 확인
with open(XAI_PATH) as f:
    code = f.read()
print('✅ 모델 경로 업데이트:')
print('   stage2_final.h5 사용 중:', 'stage2_final.h5' in code)

In [ ]:
# Streamlit 재시작
!pip install pyngrok streamlit -q

import subprocess, time
from pyngrok import ngrok

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'  # ← 토큰 입력
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

subprocess.Popen(
    ['streamlit', 'run', f'{BASE}/app/streamlit_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
)
time.sleep(4)
public_url = ngrok.connect(8501)
print('🌐', public_url)
print('\n✅ 웹캠 촬영 → REAL + 올바른 spoof type 확인하세요')

## ✅ 완료 체크리스트

| 항목 | 상태 |
|------|------|
| spoof head freeze 확인 | ⬜ |
| Fine-tuning 실행 | ⬜ |
| 웹캠 REAL 판정률 > 80% | ⬜ |
| Spoof 유형 분류 정상 (Print/Replay/Mask) | ⬜ |
| stage2_final.h5 저장 | ⬜ |
| xai_explainer.py 경로 업데이트 | ⬜ |
| Streamlit 재시작 후 최종 확인 | ⬜ |

---

### 트러블슈팅 기록 (TS-10)

| 시도 | 방법 | 결과 |
|------|------|------|
| 시도 1 | GaussianBlur + bilateralFilter | 효과 미흡 |
| 시도 2 | threshold 0.9 상향 | 효과 없음 |
| 시도 3 | 웹캠 51장 Fine-tuning (상위 30% unfreeze) | binary 98% 해결, spoof head 과적합 발생 |
| 시도 4 | binary 레이어만 가중치 이식 | shared 레이어 의존성으로 실패 |
| **시도 5** | **spoof head freeze + binary/shared만 재학습** | **✅ 최종 해결** |